# EDA + Pre-processing + Random Forest (IMDB Sentiment)

This notebook combines:
- `EDA.ipynb`
- `pre-process.ipynb`

and extends them with a complete Random Forest supervised text classification workflow required for the assignment.

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)

nltk.download('stopwords', quiet=True)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('Libraries loaded successfully.')

## 1) Dataset Overview and EDA

In [ ]:
# Load dataset
# Keep IMDB Dataset.csv in the same folder as this notebook.
df = pd.read_csv('IMDB Dataset.csv')

# Encode sentiment label for modelling
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

print('=' * 55)
print('DATASET OVERVIEW')
print('=' * 55)
print(f'Total reviews   : {len(df):,}')
print(f'Columns         : {list(df.columns)}')
print(f'Missing values  : {df.isnull().sum().sum()}')
print(f'Duplicate rows  : {df.duplicated(subset=["review"]).sum()}')
print('=' * 55)

df.head(3)

In [ ]:
# EDA: Class distribution
counts = df['sentiment'].value_counts()
pcts = df['sentiment'].value_counts(normalize=True) * 100

print('Sentiment Distribution:')
print(f"Positive : {counts['positive']:,} ({pcts['positive']:.1f}%)")
print(f"Negative : {counts['negative']:,} ({pcts['negative']:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
colors = ['#4C8BBF', '#E07B54']

bars = axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white')
axes[0].set_title('Review Count by Sentiment', fontweight='bold')
axes[0].set_ylabel('Number of Reviews')
for bar, pct in zip(bars, pcts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 150,
                 f'{bar.get_height():,}\n({pct:.1f}%)', ha='center', fontsize=9)

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Sentiment Proportion', fontweight='bold')

plt.suptitle('IMDB Dataset - Class Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# EDA: Review length analysis (word count)
df['review_length_words'] = df['review'].apply(lambda x: len(str(x).split()))

print('Review Length Statistics (words):')
print(df['review_length_words'].describe().round(2))

plt.figure(figsize=(9, 4.5))
sns.histplot(df['review_length_words'], bins=50, color='#5DADE2', edgecolor='white')
plt.title('Review Length Distribution (Word Count)', fontweight='bold')
plt.xlabel('Number of Words')
plt.ylabel('Number of Reviews')
plt.xlim(0, np.percentile(df['review_length_words'], 99))
plt.tight_layout()
plt.show()

## 2) Data Pre-processing

In [ ]:
# Remove duplicate reviews and reset index
before_rows = len(df)
df = df.drop_duplicates(subset=['review']).reset_index(drop=True)
after_rows = len(df)

print('=' * 55)
print('PRE-PROCESSING SUMMARY')
print('=' * 55)
print(f'Rows before deduplication : {before_rows:,}')
print(f'Rows after deduplication  : {after_rows:,}')
print(f'Duplicates removed        : {before_rows - after_rows:,}')
print('=' * 55)

In [ ]:
# Text cleaning for classical ML models
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def clean_text(text):
    # Remove HTML tags
    text = re.sub(r'<.*?>', ' ', str(text))
    # Keep letters only
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    # Lowercase + tokenize
    tokens = text.lower().split()
    # Remove stop words + apply stemming
    tokens = [stemmer.stem(w) for w in tokens if w not in stop_words]
    return ' '.join(tokens)

df['clean_review'] = df['review'].apply(clean_text)

print('Cleaning complete. Sample cleaned review:')
print(df['clean_review'].iloc[0][:300])

## 3) Build Supervised Model (Random Forest) - Q2.1

In [ ]:
# Train/test split and TF-IDF vectorisation
X = df['clean_review']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Compact TF-IDF to keep Random Forest training practical
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f'Training samples : {X_train_tfidf.shape[0]}')
print(f'Test samples     : {X_test_tfidf.shape[0]}')
print(f'TF-IDF features  : {X_train_tfidf.shape[1]}')

# Baseline Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_tfidf, y_train)
print('Baseline Random Forest model trained successfully.')

## 4) Evaluation Measures and Model Results - Q2.2 & Q2.3

**Selected evaluation measures:**
- **Accuracy**: overall proportion of correct predictions.
- **Precision**: percentage of predicted positive reviews that are truly positive.
- **Recall**: percentage of actual positive reviews correctly identified.
- **F1-score**: harmonic mean of precision and recall (balances both).

In [ ]:
# Baseline model evaluation
y_pred = rf_model.predict(X_test_tfidf)

acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print('=' * 55)
print('BASELINE RANDOM FOREST - EVALUATION RESULTS')
print('=' * 55)
print(f'Accuracy  : {acc:.4f} ({acc*100:.2f}%)')
print(f'Precision : {precision:.4f}')
print(f'Recall    : {recall:.4f}')
print(f'F1-Score  : {f1:.4f}')
print('=' * 55)
print('\nDetailed Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Negative', 'Positive']).plot(cmap='Blues', ax=ax)
ax.set_title('Random Forest - Confusion Matrix (Baseline)', fontweight='bold')
plt.tight_layout()
plt.show()

## 5) Hyperparameter Selection and Tuning - Q2.3

### Selected hyperparameters and rationale
- **n_estimators**: number of trees; more trees improve stability but increase runtime.
- **max_depth**: maximum depth of each tree; controls model complexity.
- **min_samples_split**: minimum samples needed to split an internal node.
- **min_samples_leaf**: minimum samples required at leaf nodes (variance control).
- **max_features**: number of features considered per split (`sqrt` / `log2`) to improve diversity.

We use **GridSearchCV** with 3-fold cross-validation and **f1_macro** scoring.

In [ ]:
# GridSearchCV for Random Forest tuning
param_grid = {
    'n_estimators': [200, 300],
    'max_depth': [None, 30],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}

rf_base = RandomForestClassifier(
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    cv=3,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

print('Running GridSearchCV...')
print(param_grid)
grid_search.fit(X_train_tfidf, y_train)

print('\nGridSearchCV complete.')
print(f'Best Parameters : {grid_search.best_params_}')
print(f'Best CV F1 Score: {grid_search.best_score_:.4f}')

In [ ]:
# Evaluate tuned model and compare with baseline
best_rf = grid_search.best_estimator_
y_pred_tuned = best_rf.predict(X_test_tfidf)

acc_t = accuracy_score(y_test, y_pred_tuned)
prec_t = precision_score(y_test, y_pred_tuned)
rec_t = recall_score(y_test, y_pred_tuned)
f1_t = f1_score(y_test, y_pred_tuned)

print('=' * 60)
print('TUNED RANDOM FOREST - EVALUATION RESULTS')
print('=' * 60)
print(f'Accuracy  : {acc_t:.4f} ({acc_t*100:.2f}%)')
print(f'Precision : {prec_t:.4f}')
print(f'Recall    : {rec_t:.4f}')
print(f'F1-Score  : {f1_t:.4f}')
print('=' * 60)
print('\nDetailed Classification Report:')
print(classification_report(y_test, y_pred_tuned, target_names=['Negative', 'Positive']))

results_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Baseline RF': [acc, precision, recall, f1],
    'Tuned RF': [acc_t, prec_t, rec_t, f1_t]
})

print('\nBaseline vs Tuned Comparison:')
print(results_df.to_string(index=False, formatters={'Baseline RF': '{:.4f}'.format, 'Tuned RF': '{:.4f}'.format}))

x = np.arange(len(results_df['Metric']))
width = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
bar1 = ax.bar(x - width/2, results_df['Baseline RF'], width, label='Baseline RF', color='steelblue')
bar2 = ax.bar(x + width/2, results_df['Tuned RF'], width, label='Tuned RF (GridSearchCV)', color='darkorange')

ax.set_ylim(0.80, 1.00)
ax.set_ylabel('Score')
ax.set_title('Random Forest - Baseline vs Tuned Performance', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(results_df['Metric'])
ax.legend()

for bar in list(bar1) + list(bar2):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.002,
            f"{bar.get_height():.4f}",
            ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 6) Critical Analysis of Predictive Model Results

- Random Forest generally handles non-linear decision boundaries well, but sparse high-dimensional TF-IDF text features often favor strong linear models such as SVM.
- If tuning improves F1-score only slightly, it suggests the baseline RF is already near its practical limit for this representation.
- A precision-recall trade-off should be interpreted in context:
  - higher precision = fewer false positives,
  - higher recall = fewer false negatives.
- Hyperparameter tuning improves control over overfitting (depth, leaf size) and model stability (number of trees).
- Further improvements can be explored using feature selection, threshold tuning, or transformer-based text embeddings.
